# 3. Data Preparation

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import os

## 3.1 Load Dataset

In [ ]:
df = pd.read_csv('../Datasets/hypertension_dataset.csv')
print(f'Total Data Awal: {df.shape}')

## 3.2 Handling Missing Values

In [ ]:
missing = df.isnull().sum()
print('Missing Values Awal:\n', missing)

# Mengisi missing value pada Medication
df['Medication'] = df['Medication'].astype('object')
df['Medication'] = df['Medication'].fillna('None')

print('\nMissing values pada Medication setelah diisi:', df['Medication'].isnull().sum())
print('\nDistribusi nilai Medication:')
print(df['Medication'].value_counts())

## 3.3 Encoding Fitur Kategorikal
Karena LightGBM dapat menangani data kategorikal secara native jika diubah menjadi integer tipe kategori, kita akan menggunakan `LabelEncoder` untuk seluruh kolom objek (termasuk target).

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'str']).columns
le_dict = {}

df_prepared = df.copy()
for col in cat_cols:
    le = LabelEncoder()
    encoded_values = le.fit_transform(df_prepared[col].astype(str))
    df_prepared[col] = pd.Series(encoded_values, index=df_prepared.index).astype('category')
    le_dict[col] = le

print("=== TABEL 1: Data Hasil Encoding ===")
display(df_prepared.head())

mapping_rows = []
for col, le in le_dict.items():
    for kategori, kode in zip(le.classes_, le.transform(le.classes_)):
        mapping_rows.append({'Fitur': col, 'Kategori Asli': kategori, 'Kode': kode})

df_mapping = pd.DataFrame(mapping_rows)

print("\n=== TABEL 2: Mapping Encoding Semua Fitur ===")
print(f"Jumlah baris tabel mapping: {len(df_mapping)}")
display(df_mapping)

## 3.4 Menyimpan Data Prepared

In [ ]:
os.makedirs('../Datasets', exist_ok=True)
df_prepared.to_csv('../Datasets/prepared_dataset.csv', index=False)
print('Data berhasil disimpan ke ../Datasets/prepared_dataset.csv')

## 3.5 Contoh Train-Test Split
Sebagai ilustrasi awal, kita akan membagi dataset menjadi 3 skenario berbeda: 70-30, 80-20, dan 90-10. Nantinya, pada tahapan *Modeling*, kita akan membandingkan performa model pada ketiga split ini.

In [ ]:
X = df_prepared.drop('Has_Hypertension', axis=1)
y = df_prepared['Has_Hypertension']

# Split 70-30
X_train_70, X_test_70, y_train_70, y_test_70 = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print('Skenario 70-30 -> X_train shape:', X_train_70.shape, '| X_test shape:', X_test_70.shape)

# Split 80-20
X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Skenario 80-20 -> X_train shape:', X_train_80.shape, '| X_test shape:', X_test_80.shape)

# Split 90-10
X_train_90, X_test_90, y_train_90, y_test_90 = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)
print('Skenario 90-10 -> X_train shape:', X_train_90.shape, '| X_test shape:', X_test_90.shape)